# 02 — Indexación en Vector Store (ChromaDB)

Carga los documentos JSON, genera embeddings multilingues y construye el vector store ChromaDB persistente. Ejecutar este notebook UNA SOLA VEZ.

In [1]:
import sys
sys.path.insert(0, '..')

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA disponible: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

PyTorch: 2.5.1+cu121
CUDA disponible: True
GPU: NVIDIA GeForce RTX 4080


In [2]:
import json
import glob
from tqdm.notebook import tqdm
from langchain_core.documents import Document

def cargar_documentos(ruta='../final_json'):
    archivos = glob.glob(f'{ruta}/**/*.json', recursive=True)
    docs = []
    for path in tqdm(archivos, desc='Cargando JSONs'):
        with open(path, encoding='utf-8') as f:
            entradas = json.load(f)
        for d in entradas:
            docs.append(Document(
                page_content=d['texto'],
                metadata={
                    'nombre_doc': d.get('nombre_doc', ''),
                    'tipo_doc':   d.get('tipo_doc', ''),
                    'seccion':    d.get('capitulo_seccion', ''),
                    'categorias': ', '.join(d.get('categoria_consumo', [])),
                    'source':     d.get('source', ''),
                    'id':         str(d.get('id', '')),
                }
            ))
    return docs

docs = cargar_documentos()
print(f'\nTotal documentos: {len(docs)}')
print(f'Ejemplo de metadata: {docs[0].metadata}')

ModuleNotFoundError: No module named 'langchain.schema'

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

EMBED_MODEL = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Usando dispositivo: {device}')

embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs={'device': device},
    encode_kwargs={'batch_size': 64},
)

vec = embeddings.embed_query('derechos del consumidor')
print(f'Dimensión del embedding: {len(vec)}')

In [ ]:
from langchain_chroma import Chroma

CHROMA_DIR = '../chroma_db'

print(f'Indexando {len(docs)} documentos en ChromaDB...')
vectorstore = Chroma.from_documents(
    docs,
    embeddings,
    persist_directory=CHROMA_DIR,
)
print(f'Vector store guardado en: {CHROMA_DIR}')

In [ ]:
# Verificar recuperación
preguntas_test = [
    'libro de reclamaciones',
    'SOAT accidente',
    'producto defectuoso garantía',
]

for q in preguntas_test:
    resultados = vectorstore.similarity_search(q, k=2)
    print(f'\nQuery: "{q}"')
    for r in resultados:
        print(f'  → {r.metadata["nombre_doc"]} | {r.metadata["seccion"][:60]}')

In [ ]:
# Búsqueda con filtro por tipo de documento
resultados_ley = vectorstore.similarity_search(
    'atención preferente adulto mayor',
    k=3,
    filter={'tipo_doc': 'ley'},
)
print('Resultados filtrados solo de leyes:')
for r in resultados_ley:
    print(f'  [{r.metadata["tipo_doc"]}] {r.metadata["nombre_doc"]}')
    print(f'  {r.page_content[:200]}...')
    print()